## 0. Configuración de rutas

In [1]:
import os

# El notebook vive en notebooks/ — subimos un nivel para llegar a la raiz del proyecto
PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATOS_PATH   = os.path.join(PROJECT_PATH, "datos")

PROC_CC = os.path.join(DATOS_PATH, "processed", "cc_news")
PROC_MIND = os.path.join(DATOS_PATH, "processed", "mind_large")
TFIDF_CC = os.path.join(DATOS_PATH, "processed", "tfidf_cc")
TFIDF_MIND = os.path.join(DATOS_PATH, "processed", "tfidf_mind")
INDEX_PATH = os.path.join(DATOS_PATH, "processed", "inverted_index")
MODELS_PATH = os.path.join(PROJECT_PATH, "models")

for path in [TFIDF_CC, TFIDF_MIND, INDEX_PATH, MODELS_PATH]:
    os.makedirs(path, exist_ok=True)

print(f"PROJECT_PATH → {PROJECT_PATH}")
print("Rutas configuradas.")

PROJECT_PATH → /Users/pm/Documents/02_UNIVERSIDAD/6TH_SEMESTER/02_DATOS MASIVOS/Proyecto/Proyecto_Datos_Masivos
Rutas configuradas.


## 1. Dependencias

In [2]:
# Ejecuta solo la primera vez
# !pip install pyspark nltk

In [3]:
import ssl
import nltk

ssl._create_default_https_context = ssl._create_unverified_context

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

STOPWORDS_EN = set(stopwords.words("english"))
print(f"Stopwords cargadas: {len(STOPWORDS_EN)}")

Stopwords cargadas: 198


## 2. Inicialización de PySpark

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

spark = (
    SparkSession.builder
    .appName("Preprocesamiento-MapReduce")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"PySpark {spark.version} listo.")

26/05/20 18:39:31 WARN Utils: Your hostname, PEDROs-Laptop.local resolves to a loopback address: 127.0.0.1; using 192.168.101.109 instead (on interface en0)
26/05/20 18:39:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 18:39:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PySpark 3.5.1 listo.


## 3. Carga de datos (salida de Fase 1)

In [5]:
cc_df = spark.read.parquet(PROC_CC)
mind_df = spark.read.parquet(PROC_MIND)

print(f"CC-News:    {cc_df.count():>8,} documentos")
print(f"MIND Large: {mind_df.count():>8,} documentos")

# Estandarizamos esquema: (doc_id, text, label)
# CC-News no tiene etiqueta -> None
cc_std = (
    cc_df
    .select(
        F.col("doc_id").cast("string"),
        F.col("text"),
        F.lit(None).cast("string").alias("label")))

# MIND tiene etiqueta de categoria
mind_std = (
    mind_df
    .select(
        F.col("news_id").alias("doc_id"),
        F.col("text"),
        F.col("category").alias("label")))

cc_std.show(2, truncate=80)
mind_std.show(2, truncate=80)

CC-News:     703,488 documentos
MIND Large:  173,550 documentos
+------+--------------------------------------------------------------------------------+-----+
|doc_id|                                                                            text|label|
+------+--------------------------------------------------------------------------------+-----+
|  1203|MIAMI (AP) — Marcell Ozuna homered to start the Miami Marlins' comeback from ...| NULL|
| 83761|ENVIRONMENTAL watchdog EcoWaste Coalition warned Cebuanos seeking fairer skin...| NULL|
+------+--------------------------------------------------------------------------------+-----+
only showing top 2 rows

+-------+--------------------------------------------------------------------------------+-----+
| doc_id|                                                                            text|label|
+-------+--------------------------------------------------------------------------------+-----+
|N104270|Kentucky Meat Shower? Yes, meat fel

## 4. MAP — Tokenización

Cada documento se convierte en una lista de tokens en minúsculas, eliminando puntuación y números.

**Paralelo al modelo MapReduce:** esta transformación es independiente por documento → se ejecuta en cada partición sin comunicación entre nodos.

In [6]:
import re

def tokenizar(texto):
    """Map: texto -> lista de tokens crudos."""
    if not texto:
        return []
    texto = texto.lower()
    texto = re.sub(r"[^a-z\s]", " ", texto)  # solo letras
    tokens = texto.split()
    return [t for t in tokens if len(t) > 2]  # descarta tokens de 1-2 chars

tokenizar_udf = F.udf(tokenizar, ArrayType(StringType()))

# Aplicar Map a ambos datasets
cc_tokens   = cc_std.withColumn("tokens_raw",   tokenizar_udf(F.col("text")))
mind_tokens = mind_std.withColumn("tokens_raw", tokenizar_udf(F.col("text")))

print("Ejemplo de tokenización:")
mind_tokens.select("doc_id", "tokens_raw").show(3, truncate=80)

Ejemplo de tokenización:
+-------+--------------------------------------------------------------------------------+
| doc_id|                                                                      tokens_raw|
+-------+--------------------------------------------------------------------------------+
|N104270|[kentucky, meat, shower, yes, meat, fell, from, the, sky, more, than, years, ...|
|N100124|[house, condemns, trump, syria, withdrawal, the, house, condemned, president,...|
|  N2248|[what, chicago, new, budget, plan, means, for, homeowners, and, housing, ther...|
+-------+--------------------------------------------------------------------------------+
only showing top 3 rows



## 5. COMBINER — Stopwords + Stemming

El Combiner reduce el volumen de datos antes del Reduce global.  
**Costo-comunicación:** sin Combiner, cada token viaja por la red. Con Combiner, solo viajan los stems relevantes.

In [7]:
stopwords_bc = spark.sparkContext.broadcast(STOPWORDS_EN)
stemmer_bc   = spark.sparkContext.broadcast(PorterStemmer())

def combiner(tokens):
    if not tokens:
        return []
    sw = stopwords_bc.value
    ps = stemmer_bc.value
    resultado = []
    for t in tokens:
        if t not in sw:
            stem = ps.stem(t)
            if len(stem) > 2:
                resultado.append(stem)
    return resultado

combiner_udf = F.udf(combiner, ArrayType(StringType()))

# Solo definimos las transformaciones — Spark no ejecuta nada todavía (lazy)
cc_limpio   = cc_tokens.withColumn("tokens", combiner_udf(F.col("tokens_raw"))).drop("tokens_raw")
mind_limpio = mind_tokens.withColumn("tokens", combiner_udf(F.col("tokens_raw"))).drop("tokens_raw")

cc_limpio   = cc_limpio.filter(F.size(F.col("tokens")) >= 5)
mind_limpio = mind_limpio.filter(F.size(F.col("tokens")) >= 5)

# El procesamiento ocurrirá al guardar Parquet.

## 6. REDUCE — TF-IDF con MLlib

**HashingTF** calcula la frecuencia de términos (TF) por documento usando hashing trick.  
**IDF** agrega el factor inverso de frecuencia de documentos — requiere una pasada global (shuffle) → aquí está el costo de comunicación del Reduce.

```
TF(t, d)  = frecuencia del término t en documento d
IDF(t)    = log( N / df(t) )   donde N = total docs, df(t) = docs que contienen t
TF-IDF    = TF * IDF
```

In [8]:
from pyspark.ml.feature import HashingTF, IDF

NUM_FEATURES = 65536  # 2^16

hashing_tf = HashingTF(inputCol="tokens", outputCol="tf_vector", numFeatures=NUM_FEATURES)
idf        = IDF(inputCol="tf_vector", outputCol="tfidf_vector", minDocFreq=3)

# Solo calculamos TF-IDF para MIND (tiene etiquetas → Fase 4)
# CC-News va directo a Fase 3 con tokens — MinHash no necesita TF-IDF
print("Calculando TF para MIND...")
tf_mind = hashing_tf.transform(mind_limpio)

print("Ajustando IDF para MIND (shuffle global)...")
idf_model_mind = idf.fit(tf_mind)
mind_tfidf = idf_model_mind.transform(tf_mind).select("doc_id", "label", "tokens", "tfidf_vector")

print("TF-IDF calculado.")
mind_tfidf.select("doc_id", "tfidf_vector").limit(2).show(truncate=80)

Calculando TF para MIND...
Ajustando IDF para MIND (shuffle global)...


TF-IDF calculado.


26/05/20 18:40:14 WARN DAGScheduler: Broadcasting large task binary with size 1077.7 KiB


+-------+--------------------------------------------------------------------------------+
| doc_id|                                                                    tfidf_vector|
+-------+--------------------------------------------------------------------------------+
|N104270|(65536,[2260,3539,9859,13003,20321,21823,21864,24707,26498,27973,34696,36456,...|
|N100124|(65536,[1737,4607,5321,10807,30034,34909,36402,37833,38823,51471,53082,55232,...|
+-------+--------------------------------------------------------------------------------+



## 7. Guardado de vectores TF-IDF

In [9]:
from pyspark.ml import PipelineModel

# Guardar tokens de CC-News (para Fase 3 — MinHash LSH)
(
    cc_limpio
    .select("doc_id", "tokens")
    .repartition(8)
    .write.mode("overwrite")
    .parquet(os.path.join(DATOS_PATH, "processed", "tokens_cc"))
)
print(f"Tokens CC-News guardados.")

# Guardar TF-IDF de MIND (para Fase 4 — MLP)
(
    mind_tfidf
    .select("doc_id", "label", "tfidf_vector")
    .repartition(4)
    .write.mode("overwrite")
    .parquet(TFIDF_MIND)
)
print(f"TF-IDF MIND guardado: {TFIDF_MIND}")

# Guardar modelo IDF (para inferencia en Fase 4)
idf_model_mind.save(os.path.join(MODELS_PATH, "idf_model_mind"))
print("Modelo IDF guardado.")

26/05/20 18:56:41 WARN DAGScheduler: Broadcasting large task binary with size 1083.4 KiB


Tokens CC-News guardados.


TF-IDF MIND guardado: /Users/pm/Documents/02_UNIVERSIDAD/6TH_SEMESTER/02_DATOS MASIVOS/Proyecto/Proyecto_Datos_Masivos/datos/processed/tfidf_mind
Modelo IDF guardado.


26/05/20 18:57:27 WARN TaskSetManager: Stage 21 contains a task of very large size (1052 KiB). The maximum recommended task size is 1000 KiB.


In [10]:
# Guardar índice invertido como Parquet (token → lista de doc_ids)
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

schema_indice = StructType([
    StructField("token",   StringType(),             False),
    StructField("doc_ids", ArrayType(StringType()),  False),
])

indice_df = spark.createDataFrame(
    indice_invertido.map(lambda x: (x[0], x[1])),
    schema=schema_indice)

indice_df.write.mode("overwrite").parquet(INDEX_PATH)
print(f"Índice invertido guardado: {INDEX_PATH}")
print(f"Términos únicos indexados: {indice_df.count():,}")

NameError: name 'indice_invertido' is not defined

## 9. Análisis costo-comunicación — Fase 2

In [ ]:
n_docs  = mind_limpio.count()
n_terms = indice_df.count()

print("MODELO COSTO-COMUNICACIÓN — PREPROCESAMIENTO (Fase 2)")
print("-" * 55)
print(f"Documentos procesados : {n_docs:,}")
print(f"Términos únicos (vocab): {n_terms:,}")
print(f"Dimensión vector TF-IDF: {NUM_FEATURES:,} (hashing trick)")
print()
print("Costo por etapa:")
print(f"  Map (tokenizar):      O(n · L)   — L = longitud media del documento")
print(f"  Combiner (stem/stop): O(n · L)   — local, sin comunicación")
print(f"  Reduce TF:            O(n · L)   — local por partición")
print(f"  Reduce IDF (shuffle): O(|V| · p) — |V|=vocab, p=particiones (COSTOSO)")
print(f"  Índice invertido:     O(n · L)   — 1 shuffle groupByKey")
print()
print("Optimizaciones aplicadas:")
print("  - Broadcast de stopwords: evita transferir la lista a cada tarea")
print("  - minDocFreq=3 en IDF: elimina términos raros → reduce |V| efectivo")
print("  - HashingTF: evita diccionario global (sin comunicación en Map)")

In [ ]:
spark.stop()
print("SparkSession cerrada. Fase 2 completada.")
print("Siguiente paso: 03_lsh_dedup.ipynb")